# Localization: Plot Builder

Interactive plot builder for localization experiments.
Generates matplotlib previews and copy-pasteable LaTeX
(pgfplots) source for the paper.

## Metrics

- **accuracy_primary**: the model picks which sentence was
  perturbed, using the first-token logit for each label
  (e.g. logit of " A" vs logit of " B"). Correct if
  the highest-logit label matches the ground truth.
- **accuracy_aggregate**: same idea, but uses the summed
  probability across all token variants of each label.
- **content_accuracy_primary**: for controls only. The model
  picks which sentence matches a topic (e.g. "about animals").
  Uses primary (first-token) logits.

## Error bands

All experiments are binary classification ($n$ Bernoulli
trials). The error band can toggle between:

- **SE** (standard error): $\sqrt{\hat{p}(1-\hat{p})/n}$.
  Shows uncertainty of the accuracy estimate.
- **SD** (standard deviation): $\sqrt{\hat{p}(1-\hat{p})}$.
  Shows the spread of individual binary outcomes. This band
  is much wider (it does not shrink with $n$).

In [ ]:
import os
import pathlib

_this_dir = pathlib.Path(os.path.abspath("")).resolve()
if _this_dir.name == "paper":
    os.chdir(_this_dir.parent)

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from wandb_cache import load_sweep

In [ ]:
PROJECT = "llm-mechanistic-detection"

# --- Sweep IDs ---
LOC_MAIN = {"dropout": "zjm2ug9y", "noise": "1lsya445"}
LOC_CTRL = {"dropout": "unnbhcwb", "noise": "vay012s7"}
LOC_CTRL_NOISE_EXT = ["l0ugcqlz", "owsfjyp4"]

# --- Columns ---
COL_MODEL = "model"
COL_DROPOUT = "perturbation.dropout_rate"
COL_NOISE = "perturbation.noise_std"
COL_SENTENCES = "experiment.sentences_file"
COL_PROMPT = "prompt_turns"

# --- Models (no Gemma) ---
MODELS = ["llama3_8b", "qwen3_14b", "qwen3_32b", "olmo3_32b"]

MODEL_COLORS_MPL = {
    "llama3_8b": "#f4b6c2",
    "qwen3_14b": "#8fae82",
    "qwen3_32b": "#1b3a6b",
    "olmo3_32b": "#8b1a1a",
}
MODEL_COLORS_LATEX = {
    "llama3_8b": "red!40",
    "qwen3_14b": "green!40!black!30",
    "qwen3_32b": "blue!70!black",
    "olmo3_32b": "red!70!black",
}

DROPOUT_XLABEL = {"mpl": r"Dropout rate $p$", "latex": "Dropout rate $p$"}
NOISE_XLABEL = {"mpl": r"Noise SD $\sigma$", "latex": "Noise SD $\\sigma$"}

METRICS = [
    "accuracy_primary",
    "accuracy_aggregate",
    "accuracy_argmax",
    "content_accuracy_primary",
    "content_accuracy_aggregate",
    "roc_auc_primary",
    "roc_auc_aggregate",
    "macro_f1_primary",
    "macro_f1_aggregate",
    "entropy_mean",
]

In [ ]:
# --- Load all data ---
print("Main sweeps:")
df_main_d = load_sweep(LOC_MAIN["dropout"], PROJECT)
df_main_n = load_sweep(LOC_MAIN["noise"], PROJECT)
df_main_d = df_main_d[df_main_d[COL_MODEL].isin(MODELS)].copy()
df_main_n = df_main_n[df_main_n[COL_MODEL].isin(MODELS)].copy()

print("\nControl sweeps:")
df_ctrl_d = load_sweep(LOC_CTRL["dropout"], PROJECT)
df_ctrl_n = load_sweep(LOC_CTRL["noise"], PROJECT)
df_ctrl_d = df_ctrl_d[df_ctrl_d[COL_MODEL].isin(MODELS)].copy()
df_ctrl_n = df_ctrl_n[df_ctrl_n[COL_MODEL].isin(MODELS)].copy()

# Extension sweeps (finer noise granularity for some models)
print("\nControl noise extensions:")
ext_dfs = []
for sid in LOC_CTRL_NOISE_EXT:
    ext = load_sweep(sid, PROJECT)
    ext = ext[ext[COL_MODEL].isin(MODELS)]
    ext_dfs.append(ext)
    print(f"  {sid}: {len(ext)} runs ({sorted(ext[COL_MODEL].unique())})")
df_ctrl_n = pd.concat([df_ctrl_n] + ext_dfs, ignore_index=True)

# Samples per run
N_PER_RUN = (
    int(df_main_d["total_samples"].mode().iloc[0])
    if "total_samples" in df_main_d.columns
    else 1000
)
print(f"\nSamples per run: {N_PER_RUN}")


# Discover grouping values
def _stem(path):
    return pathlib.PurePosixPath(path).stem


sent_files = sorted(df_main_d[COL_SENTENCES].unique())
sent_map = {_stem(sf): sf for sf in sent_files}
print(f"Sentence lengths: {list(sent_map.keys())}")

prompt_files = sorted(df_ctrl_d[COL_PROMPT].unique())
prompt_map = {p.split("/")[-1]: p for p in prompt_files}
print(f"Control prompts: {list(prompt_map.keys())}")

avail_main = [m for m in METRICS if m in df_main_d.columns]
avail_ctrl = [m for m in METRICS if m in df_ctrl_d.columns]
print(f"Metrics (main): {avail_main}")
print(f"Metrics (ctrl): {avail_ctrl}")

In [ ]:
# ── Curve computation ────────────────────────────────────────


def compute_curves(df, x_col, metric, group_col, group_value, n_per_run, models):
    """Return {model: (x, y_pct, se_pct, sd_pct)}.

    group_value == "all" pools across all values of group_col.
    For accuracy-family metrics, SE and SD come from the
    Bernoulli formula.  For others, they are empirical across
    runs in the group.
    """
    is_pct = "accuracy" in metric or "f1" in metric
    scale = 100 if is_pct else 1
    curves = {}
    for model in models:
        sub = df[df[COL_MODEL] == model]
        if metric not in sub.columns or sub.empty:
            continue
        if group_value != "all":
            sub = sub[sub[group_col] == group_value]
        if sub.empty:
            continue

        grp = sub.groupby(x_col)[metric]
        avg = grp.mean().sort_index() * scale
        if avg.empty:
            continue

        if is_pct:
            p = avg / 100
            pq = p * (1 - p)
            n_total = grp.count() * n_per_run
            se = np.sqrt(pq / n_total) * 100
            sd = np.sqrt(pq) * 100
        else:
            se = grp.sem().sort_index() * scale
            sd = grp.std().sort_index() * scale

        curves[model] = (avg.index.values, avg.values, se.values, sd.values)
    return curves


# ── Matplotlib ───────────────────────────────────────────────


def plot_panel(
    ax, curves, x_label, metric, error_type, ymin, ymax, is_right=False, ref_lines=None
):
    """ref_lines: list of y-values for gray dashed reference lines."""
    is_pct = "accuracy" in metric or "f1" in metric

    for model in MODELS:
        if model not in curves:
            continue
        x, y, se, sd = curves[model]
        err = se if error_type == "SE" else sd
        c = MODEL_COLORS_MPL[model]
        ax.plot(x, y, marker="o", markersize=3, linewidth=2, color=c, label=model)
        ax.fill_between(x, y - err, y + err, alpha=0.3, color=c)

    if is_pct:
        ax.axhline(50, color="gray", ls="--", alpha=0.5, label="chance")
    for rl in ref_lines or []:
        ax.axhline(rl, color="gray", ls="--", alpha=0.5)
    ax.set_xlabel(x_label)
    if is_right:
        ax.set_ylabel("")
        ax.set_yticklabels([])
    else:
        ax.set_ylabel("Accuracy (%)" if is_pct else metric)
    ax.grid(True, alpha=0.3)
    if ymin is not None and ymax is not None and ymin < ymax:
        ax.set_ylim(ymin, ymax)


# ── LaTeX / pgfplots ────────────────────────────────────────


def latex_panel(
    curves,
    x_label,
    metric,
    error_type,
    models,
    is_right=False,
    ref_lines=None,
    ymin=None,
    ymax=None,
):
    is_pct = "accuracy" in metric or "f1" in metric
    all_x = (
        np.concatenate([c[0] for c in curves.values()]) if curves else np.array([0, 1])
    )
    xmin, xmax = float(all_x.min()), float(all_x.max())

    L = []
    L.append(r"\begin{tikzpicture}")
    L.append(r"\begin{axis}[")
    L.append("    height=7cm,")
    L.append("    width=\\linewidth,")
    L.append("    grid=major,")
    L.append(f"    xlabel={{{x_label}}},")
    if is_right:
        L.append("    ylabel={},")
        L.append("    yticklabels={},")
    else:
        ylabel = "Accuracy (\\%)" if is_pct else metric
        L.append(f"    ylabel={{{ylabel}}},")
    L.append(f"    xmin={xmin},")
    L.append(f"    xmax={xmax},")
    if ymin is not None and ymax is not None:
        L.append(f"    ymin={ymin}, ymax={ymax},")
    L.append("    legend entries={},")
    L.append("]")

    for model in models:
        if model not in curves:
            continue
        x, y, se, sd = curves[model]
        err = se if error_type == "SE" else sd
        c = MODEL_COLORS_LATEX[model]

        up = " ".join(f"({xi:.4f},{yi + ei:.2f})" for xi, yi, ei in zip(x, y, err))
        lo = " ".join(
            f"({xi:.4f},{yi - ei:.2f})" for xi, yi, ei in reversed(list(zip(x, y, err)))
        )
        L.append(
            f"\\addplot[{c}, fill={c}, fill opacity=0.3, "
            f"draw=none, forget plot] "
            f"coordinates {{{up} {lo}}} --cycle;"
        )
        coords = " ".join(f"({xi:.4f},{yi:.2f})" for xi, yi in zip(x, y))
        L.append(
            f"\\addplot[{c}, mark=o, mark size=1, "
            f"line width=1pt, forget plot] "
            f"coordinates {{{coords}}};"
        )

    if is_pct:
        L.append(
            f"\\addplot[gray, dashed, line width=0.5pt, "
            f"forget plot] coordinates "
            f"{{({xmin},50) ({xmax},50)}};"
        )
    for rl in ref_lines or []:
        L.append(
            f"\\addplot[gray, dashed, line width=0.5pt, "
            f"forget plot] coordinates "
            f"{{({xmin},{rl}) ({xmax},{rl})}};"
        )
    L.append(r"\end{axis}")
    L.append(r"\end{tikzpicture}")
    return "\n".join(L)


def latex_figure(
    curves_d,
    curves_n,
    metric,
    error_type,
    models,
    caption,
    label,
    ref_lines=None,
    ymin=None,
    ymax=None,
):
    d_tex = latex_panel(
        curves_d,
        DROPOUT_XLABEL["latex"],
        metric,
        error_type,
        models,
        ref_lines=ref_lines,
        ymin=ymin,
        ymax=ymax,
    )
    n_tex = latex_panel(
        curves_n,
        NOISE_XLABEL["latex"],
        metric,
        error_type,
        models,
        is_right=True,
        ref_lines=ref_lines,
        ymin=ymin,
        ymax=ymax,
    )

    items = []
    for m in models:
        if m in curves_d or m in curves_n:
            c = MODEL_COLORS_LATEX[m]
            ml = m.replace("_", r"\_")
            items.append(
                f"\\tikz\\draw[{c}, thick, mark=o, "
                f"mark size=1.5] plot coordinates "
                f"{{(0,0) (0.4,0)}}; {ml}"
            )
    legend = "\\hspace{1em}".join(items)
    err_tag = "SE" if error_type == "SE" else "SD"

    return (
        "\\begin{figure}[ht]\n"
        "\\centering\n"
        f"{legend}\n"
        "\\\\[6pt]\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {d_tex}\n"
        f"\\end{{minipage}}\n"
        "\\hfill\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {n_tex}\n"
        f"\\end{{minipage}}\n"
        f"\\caption{{{caption} "
        f"Bands show $\\pm${err_tag}.}}\n"
        f"\\label{{fig:{label}}}\n"
        "\\end{figure}"
    )

## Main Experiments

Accuracy at detecting which sentence was perturbed, as a
function of perturbation strength. One curve per model.
Select a specific sentence length or pool across all six.

In [ ]:
def make_interactive(
    df_d, df_n, group_col, group_map, avail_metrics, section_label, ref_lines=None
):
    """Build a widget panel for one experiment section."""
    group_opts = [("all (pooled)", "all")] + [
        (label, val) for label, val in group_map.items()
    ]

    w_group = widgets.Dropdown(options=group_opts, value="all", description="Group:")
    w_metric = widgets.Dropdown(
        options=avail_metrics, value=avail_metrics[0], description="Metric:"
    )
    w_error = widgets.RadioButtons(
        options=["SE", "SD"],
        value="SE",
        description="Band:",
        layout=widgets.Layout(width="auto"),
    )
    w_models = {
        m: widgets.Checkbox(value=True, description=m, indent=False) for m in MODELS
    }
    w_ymin = widgets.FloatText(
        value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
    )
    w_ymax = widgets.FloatText(
        value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
    )
    out = widgets.Output()

    def redraw(*_):
        out.clear_output(wait=True)
        with out:
            gv = w_group.value
            metric = w_metric.value
            et = w_error.value
            models = [m for m in MODELS if w_models[m].value]
            ymin = w_ymin.value if not np.isnan(w_ymin.value) else None
            ymax = w_ymax.value if not np.isnan(w_ymax.value) else None

            cd = compute_curves(
                df_d, COL_DROPOUT, metric, group_col, gv, N_PER_RUN, models
            )
            cn = compute_curves(
                df_n, COL_NOISE, metric, group_col, gv, N_PER_RUN, models
            )

            # --- matplotlib preview ---
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
            plot_panel(
                ax1,
                cd,
                DROPOUT_XLABEL["mpl"],
                metric,
                et,
                ymin,
                ymax,
                ref_lines=ref_lines,
            )
            plot_panel(
                ax2,
                cn,
                NOISE_XLABEL["mpl"],
                metric,
                et,
                ymin,
                ymax,
                is_right=True,
                ref_lines=ref_lines,
            )

            handles, labels = ax1.get_legend_handles_labels()
            fig.legend(
                handles,
                labels,
                loc="upper center",
                ncol=max(len(models), 1),
                fontsize=10,
                bbox_to_anchor=(0.5, 1.08),
            )

            gl = (
                "pooled"
                if gv == "all"
                else (
                    _stem(gv) if "/" in str(gv) or "." in str(gv) else gv.split("/")[-1]
                )
            )
            fig.suptitle(
                f"{section_label} ({gl}, \u00b1{et})",
                fontsize=14,
                y=1.12,
            )
            fig.tight_layout()
            plt.show()

            # --- LaTeX ---
            caption = f"{section_label} accuracy ({gl})."
            safe = gl.replace("_", "-")
            label_tag = f"loc-{section_label.lower().split()[0]}-{safe}"
            print(
                latex_figure(
                    cd,
                    cn,
                    metric,
                    et,
                    models,
                    caption,
                    label_tag,
                    ref_lines=ref_lines,
                    ymin=ymin,
                    ymax=ymax,
                )
            )

    for w in [w_group, w_metric, w_error, w_ymin, w_ymax]:
        w.observe(redraw, names="value")
    for cb in w_models.values():
        cb.observe(redraw, names="value")

    display(
        widgets.HBox([w_group, w_metric, w_error]),
        widgets.HBox([w_ymin, w_ymax]),
        widgets.HBox(list(w_models.values())),
        out,
    )
    redraw()


make_interactive(df_main_d, df_main_n, COL_SENTENCES, sent_map, avail_main, "Main")

## Control Experiments

Controls use topic based prompts ("which sentence is about
animals?") instead of perturbation detection. There are
5 prompt pairs per (model, perturbation strength).

Extension noise data (finer granularity for llama, olmo32,
qwen32) is already merged into the noise DataFrame above.

In [ ]:
make_interactive(
    df_ctrl_d, df_ctrl_n, COL_PROMPT, prompt_map, avail_ctrl, "Control", ref_lines=[95]
)

## Per-model token count breakdown

Same main experiment data, but one model at a time with a
separate curve per sentence length. Color goes from light
pastel (short sentences) to dark (long sentences).

In [ ]:
SORTED_TOKS = sorted(sent_map.keys(), key=lambda s: int(s.replace("tok", "")))
TOK_COLORS = {
    "3tok": "#a8d8ea",  # light sky blue
    "7tok": "#f4b6c2",  # soft pink
    "11tok": "#b5e6a3",  # light green
    "15tok": "#f0c75e",  # warm gold
    "19tok": "#c4a4e0",  # lavender
    "23tok": "#e07b54",  # terracotta
}
TOK_COLORS_LATEX = {
    "3tok": "tokThree",
    "7tok": "tokSeven",
    "11tok": "tokEleven",
    "15tok": "tokFifteen",
    "19tok": "tokNineteen",
    "23tok": "tokTwentythree",
}
TOK_DEFINECOLORS = (
    "\\definecolor{tokThree}{HTML}{A8D8EA}\n"
    "\\definecolor{tokSeven}{HTML}{F4B6C2}\n"
    "\\definecolor{tokEleven}{HTML}{B5E6A3}\n"
    "\\definecolor{tokFifteen}{HTML}{F0C75E}\n"
    "\\definecolor{tokNineteen}{HTML}{C4A4E0}\n"
    "\\definecolor{tokTwentythree}{HTML}{E07B54}"
)


def _compute_tok_curves(df, x_col, metric, model):
    """Return {tok_label: (x, y, se, sd)} for a single model."""
    is_pct = "accuracy" in metric or "f1" in metric
    scale = 100 if is_pct else 1
    curves = {}
    sub_model = df[df[COL_MODEL] == model]
    if metric not in sub_model.columns or sub_model.empty:
        return curves
    for tok_label in SORTED_TOKS:
        sub = sub_model[sub_model[COL_SENTENCES] == sent_map[tok_label]]
        if sub.empty:
            continue
        grp = sub.groupby(x_col)[metric]
        avg = grp.mean().sort_index() * scale
        if avg.empty:
            continue
        if is_pct:
            p = avg / 100
            pq = p * (1 - p)
            n_total = grp.count() * N_PER_RUN
            se = np.sqrt(pq / n_total) * 100
            sd = np.sqrt(pq) * 100
        else:
            se = grp.sem().sort_index() * scale
            sd = grp.std().sort_index() * scale
        curves[tok_label] = (avg.index.values, avg.values, se.values, sd.values)
    return curves


def _plot_tok_panel(
    ax, curves, x_label, metric, error_type, ymin, ymax, is_right=False
):
    is_pct = "accuracy" in metric or "f1" in metric
    for tok_label in SORTED_TOKS:
        if tok_label not in curves:
            continue
        x, y, se, sd = curves[tok_label]
        err = se if error_type == "SE" else sd
        c = TOK_COLORS[tok_label]
        ax.plot(x, y, marker="o", markersize=3, linewidth=2, color=c, label=tok_label)
        ax.fill_between(x, y - err, y + err, alpha=0.2, color=c)

    if is_pct:
        ax.axhline(50, color="gray", ls="--", alpha=0.5)
    ax.set_xlabel(x_label)
    if is_right:
        ax.set_ylabel("")
        ax.set_yticklabels([])
    else:
        ax.set_ylabel("Accuracy (%)" if is_pct else metric)
    ax.grid(True, alpha=0.3)
    if ymin is not None and ymax is not None and ymin < ymax:
        ax.set_ylim(ymin, ymax)


def _latex_tok_panel(
    curves, x_label, metric, error_type, is_right=False, ymin=None, ymax=None
):
    is_pct = "accuracy" in metric or "f1" in metric
    all_x = (
        np.concatenate([c[0] for c in curves.values()]) if curves else np.array([0, 1])
    )
    xmin, xmax = float(all_x.min()), float(all_x.max())

    L = []
    L.append(r"\begin{tikzpicture}")
    L.append(r"\begin{axis}[")
    L.append("    height=7cm,")
    L.append("    width=\\linewidth,")
    L.append("    grid=major,")
    L.append(f"    xlabel={{{x_label}}},")
    if is_right:
        L.append("    ylabel={},")
        L.append("    yticklabels={},")
    else:
        ylabel = "Accuracy (\\%)" if is_pct else metric
        L.append(f"    ylabel={{{ylabel}}},")
    L.append(f"    xmin={xmin},")
    L.append(f"    xmax={xmax},")
    if ymin is not None and ymax is not None:
        L.append(f"    ymin={ymin}, ymax={ymax},")
    L.append("    legend entries={},")
    L.append("]")

    for tok_label in SORTED_TOKS:
        if tok_label not in curves:
            continue
        x, y, se, sd = curves[tok_label]
        err = se if error_type == "SE" else sd
        c = TOK_COLORS_LATEX[tok_label]

        up = " ".join(f"({xi:.4f},{yi + ei:.2f})" for xi, yi, ei in zip(x, y, err))
        lo = " ".join(
            f"({xi:.4f},{yi - ei:.2f})" for xi, yi, ei in reversed(list(zip(x, y, err)))
        )
        L.append(
            f"\\addplot[{c}, fill={c}, fill opacity=0.3, "
            f"draw=none, forget plot] "
            f"coordinates {{{up} {lo}}} --cycle;"
        )
        coords = " ".join(f"({xi:.4f},{yi:.2f})" for xi, yi in zip(x, y))
        L.append(
            f"\\addplot[{c}, mark=o, mark size=1, "
            f"line width=1pt, forget plot] "
            f"coordinates {{{coords}}};"
        )

    if is_pct:
        L.append(
            f"\\addplot[gray, dashed, line width=0.5pt, "
            f"forget plot] coordinates "
            f"{{({xmin},50) ({xmax},50)}};"
        )
    L.append(r"\end{axis}")
    L.append(r"\end{tikzpicture}")
    return "\n".join(L)


def _latex_tok_figure(
    curves_d, curves_n, metric, error_type, model, ymin=None, ymax=None
):
    d_tex = _latex_tok_panel(
        curves_d, DROPOUT_XLABEL["latex"], metric, error_type, ymin=ymin, ymax=ymax
    )
    n_tex = _latex_tok_panel(
        curves_n,
        NOISE_XLABEL["latex"],
        metric,
        error_type,
        is_right=True,
        ymin=ymin,
        ymax=ymax,
    )

    items = []
    for t in SORTED_TOKS:
        if t in curves_d or t in curves_n:
            c = TOK_COLORS_LATEX[t]
            items.append(
                f"\\tikz\\draw[{c}, thick, mark=o, "
                f"mark size=1.5] plot coordinates "
                f"{{(0,0) (0.4,0)}}; {t}"
            )
    legend = "\\hspace{1em}".join(items)
    err_tag = "SE" if error_type == "SE" else "SD"
    model_safe = model.replace("_", r"\_")

    return (
        f"{TOK_DEFINECOLORS}\n"
        "\\begin{figure}[ht]\n"
        "\\centering\n"
        f"{legend}\n"
        "\\\\[6pt]\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {d_tex}\n"
        f"\\end{{minipage}}\n"
        "\\hfill\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {n_tex}\n"
        f"\\end{{minipage}}\n"
        f"\\caption{{Token count breakdown for {model_safe}. "
        f"Bands show $\\pm${err_tag}.}}\n"
        f"\\label{{fig:loc-tok-{model}}}\n"
        "\\end{figure}"
    )


def make_tok_interactive():
    w_model = widgets.Dropdown(options=MODELS, value=MODELS[0], description="Model:")
    w_metric = widgets.Dropdown(
        options=avail_main, value=avail_main[0], description="Metric:"
    )
    w_error = widgets.RadioButtons(
        options=["SE", "SD"],
        value="SE",
        description="Band:",
        layout=widgets.Layout(width="auto"),
    )
    w_ymin = widgets.FloatText(
        value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
    )
    w_ymax = widgets.FloatText(
        value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
    )
    out = widgets.Output()

    def redraw(*_):
        out.clear_output(wait=True)
        with out:
            model = w_model.value
            metric = w_metric.value
            et = w_error.value
            ymin = w_ymin.value if not np.isnan(w_ymin.value) else None
            ymax = w_ymax.value if not np.isnan(w_ymax.value) else None

            cd = _compute_tok_curves(df_main_d, COL_DROPOUT, metric, model)
            cn = _compute_tok_curves(df_main_n, COL_NOISE, metric, model)

            # --- matplotlib preview ---
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
            _plot_tok_panel(ax1, cd, DROPOUT_XLABEL["mpl"], metric, et, ymin, ymax)
            _plot_tok_panel(
                ax2, cn, NOISE_XLABEL["mpl"], metric, et, ymin, ymax, is_right=True
            )

            handles, labels = ax1.get_legend_handles_labels()
            fig.legend(
                handles,
                labels,
                loc="upper center",
                ncol=len(SORTED_TOKS),
                fontsize=10,
                bbox_to_anchor=(0.5, 1.08),
            )
            fig.suptitle(
                f"Token count breakdown \u2014 {model} (\u00b1{et})",
                fontsize=14,
                y=1.12,
            )
            fig.tight_layout()
            plt.show()

            # --- LaTeX ---
            print(_latex_tok_figure(cd, cn, metric, et, model, ymin=ymin, ymax=ymax))

    for w in [w_model, w_metric, w_error, w_ymin, w_ymax]:
        w.observe(redraw, names="value")

    display(
        widgets.HBox([w_model, w_metric, w_error]),
        widgets.HBox([w_ymin, w_ymax]),
        out,
    )
    redraw()


make_tok_interactive()